# eBiblio Book Loan Automation (Google Colab)

This notebook prepares and runs `book-loan.robot` in a Google Colab runtime. Run the cells from top to bottom. You will upload the Robot suite and your private `credentials.resource` file; the notebook does not store credentials in its source.

> **Live action warning:** The live run logs in to eBiblio and may place a reservation or complete a real loan. It can wait up to four hours for today's newspaper. The live run is disabled by default; review the workflow and enable the explicit confirmation only when you intend to proceed. Keep credentials and generated reports private.

### 1. Mount Google Drive
Mount your Google Drive to use it as storage for your configuration and report files.

In [ ]:
from google.colab import drive
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

# Define the working directory inside Google Drive
WORK_DIR = Path('/content/drive/MyDrive/ebiblio-book-loan')
WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"Using Google Drive directory: {WORK_DIR}")

## 2. Upload the Robot suite and credentials

Upload both `book-loan.robot` and your own `credentials.resource` when prompted. The credentials file must define non-empty `${USUARIO}` and `${PASSWORD}` variables. The cell writes the files into the temporary Colab runtime and does not display their contents.

In [ ]:
import urllib.request
from pathlib import Path
import os
import re

# ==================== PARÁMETROS FORMULARIO COLAB ====================
#@markdown ### Configuración manual de GitHub (si falla la detección automática):
GITHUB_OWNER = "mrg321" #@param {type:"string"}
GITHUB_REPO = "autobiblio" #@param {type:"string"}
GITHUB_BRANCH = "main" #@param {type:"string"}
# =====================================================================

# Forzamos que la variable WORK_DIR apunte a Google Drive de forma persistente
WORK_DIR = Path('/content/drive/MyDrive/ebiblio-book-loan')
WORK_DIR.mkdir(parents=True, exist_ok=True)

def obtener_repo_desde_notebook():
    """Intenta recuperar el repositorio de GitHub si el notebook se cargó de ahí."""
    try:
        from google.colab import _message
        nb_json = _message.blocking_request('get_ipynb', request='', timeout_sec=5)
        if nb_json and 'metadata' in nb_json:
            colab_meta = nb_json['metadata'].get('colab', {})
            repo_history = colab_meta.get('provenance', [])
            for p in repo_history:
                if 'github.com' in p.get('url', ''):
                    return p['url']
    except Exception:
        pass
    return None

# Intentamos autodetectar el origen
repo_detectado = obtener_repo_desde_notebook()

owner = GITHUB_OWNER
repo = GITHUB_REPO
branch = GITHUB_BRANCH

if repo_detectado:
    match = re.search(r"github\.com/([^/]+)/([^/]+)/(?:blob|raw)/([^/]+)/", repo_detectado)
    if match:
        owner, repo, branch = match.groups()
        print(f"✓ Repositorio autodetectado con éxito: {owner}/{repo} (rama: {branch})")
    else:
        print("⚠ Se detectó una URL de GitHub pero no se pudo parsear automáticamente.")
else:
    print("ℹ No se pudo autodetectar el cuaderno de forma directa. Utilizando los valores del formulario de la derecha.")

# Construimos las URLs raw dinámicamente
GITHUB_ROBOT_URL = f"https://raw.githubusercontent.com/{owner}/{repo}/{branch}/book-loan.robot"
GITHUB_CREDENTIALS_URL = f"https://raw.githubusercontent.com/{owner}/{repo}/{branch}/credentials.resource"

def descargar_desde_github(url, destino):
    try:
        print(f"Descargando desde GitHub: {url}...")
        req = urllib.request.Request(
            url,
            headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
        )
        with urllib.request.urlopen(req) as response, open(destino, 'wb') as out_file:
            out_file.write(response.read())
        print(f"✓ Archivo guardado con éxito en: {destino}")
    except Exception as e:
        raise RuntimeError(
            f"No se pudo descargar el archivo desde GitHub.\n"
            f"Ruta intentada: {url}\n"
            f"Error obtenido: {e}\n\n"
            f"Por favor, verifica los campos de configuración en el formulario (usuario, repositorio y rama)."
        )

# Definir las rutas locales de destino en Google Drive
robot_path = WORK_DIR / "book-loan.robot"
credentials_path = WORK_DIR / "credentials.resource"

# Ejecutar las descargas automáticas
descargar_desde_github(GITHUB_ROBOT_URL, robot_path)
descargar_desde_github(GITHUB_CREDENTIALS_URL, credentials_path)

# Ajustar permisos de seguridad del archivo de credenciales
os.chmod(credentials_path, 0o600)

# Validación rápida de estructura del Robot suite
robot_source = robot_path.read_text(encoding="utf-8")
if not re.search(r"(?mi)^\s*Resource\s+credentials\.resource\s*$", robot_source):
    raise ValueError("El archivo book-loan.robot descargado debe importar credentials.resource")

# Validación rápida de variables en las Credenciales
credentials_source = credentials_path.read_text(encoding="utf-8")
for variable in ("USUARIO", "PASSWORD"):
    if not re.search(rf"(?m)^\$\{{{variable}\}}\s+\S", credentials_source):
        raise ValueError(
            f"El archivo credentials.resource debe definir la variable ${{{variable}}} con un valor válido."
        )

print(f"\n¡Completado! Ambos archivos se han descargado e instalado de forma persistente en: {WORK_DIR}")

## 3. Check and install the pinned Python dependencies

The project pins Robot Framework and Browser Library in `requirements.txt`. This cell checks the runtime first and installs only missing or mismatched versions.

In [ ]:
import importlib.metadata
import sys
from IPython import get_ipython

required_versions = {
    "robotframework": "7.5",
    "robotframework-browser": "20.6.0",
}
packages_to_install = []
for package, required_version in required_versions.items():
    try:
        installed_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed_version = None
    print(f"{package}: {installed_version or 'not installed'} (required {required_version})")
    if installed_version != required_version:
        packages_to_install.append(f"{package}=={required_version}")

if packages_to_install:
    get_ipython().run_line_magic("pip", "install " + " ".join(packages_to_install))
else:
    print("Pinned versions are already installed.")

for package, required_version in required_versions.items():
    installed_version = importlib.metadata.version(package)
    if installed_version != required_version:
        raise RuntimeError(
            f"{package} {installed_version} is installed, but {required_version} is required."
        )
print("Python dependencies are ready.")

## 4. Prepare Node.js for Browser Library

Browser Library uses Node.js. Google Colab normally includes it; install the system packages only if either `node` or `npm` is missing.

In [ ]:
import shutil
import subprocess
import sys

if not sys.platform.startswith("linux"):
    raise RuntimeError("Use a Linux-based Google Colab runtime for this notebook.")

if shutil.which("node") is None or shutil.which("npm") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "nodejs", "npm"], check=True)

for executable in ("node", "npm"):
    version = subprocess.run(
        [executable, "--version"], check=True, capture_output=True, text=True
    ).stdout.strip()
    print(f"{executable}: {version}")

## 5. Install shared libraries

In [ ]:
import subprocess
print("Instalando las librerías compartidas del sistema requeridas por Playwright/Chromium...")
try:
    # Ejecuta el comando oficial de Playwright para instalar dependencias de sistema en Ubuntu
    subprocess.run(["pip", "install", "playwright"], check=True)
    subprocess.run(["playwright", "install-deps"], check=True)
    print("\n✓ ¡Librerías del sistema instaladas correctamente!")
except Exception as e:
    print(f"\n⚠ Error instalando dependencias del sistema: {e}")
    print("Intentando instalación manual alternativa...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run([
        "apt-get", "install", "-y", "-qq",
        "libatk-1.0-0", "libatk-bridge2.0-0", "libxkbcommon0", "libxcomposite1",
        "libxdamage1", "libxrandr2", "libgbm1", "libasound2", "libpango-1.0-0",
        "libcairo2", "libatspi0"
    ], check=True)
    print("\n✓ ¡Librerías del sistema instaladas manualmente!")

## 6. Install Chromium for Browser Library

This initializes the Chromium browser that the Robot suite will use in headless mode.

In [ ]:
subprocess.run(
    ["rfbrowser", "init", "chromium"],
    cwd=WORK_DIR,
    check=True,
)
print("Chromium is ready for the Robot Framework Browser Library.")

## 7. Validate without logging in

The dry run checks suite parsing, resource imports, and keyword resolution. It does not open a browser or submit a loan/reservation.

In [ ]:
dry_run_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "robot",
        "--dryrun",
        "--output",
        "NONE",
        "--log",
        "NONE",
        "--report",
        "NONE",
        "--variable",
        "HEADLESS:True",
        str(WORK_DIR / "book-loan.robot"),
    ],
    cwd=WORK_DIR,
    check=False,
)
if dry_run_result.returncode != 0:
    raise RuntimeError(
        f"Robot Framework dry run failed with exit code {dry_run_result.returncode}. "
        "Resolve the reported issue before enabling the live run."
    )
print("Dry run passed; no live browser actions were performed.")

## 8. Optional live loan/reservation attempt

Only change `RUN_LIVE_BOOK_LOAN` to `True` after reviewing this Robot workflow and confirming that you want it to log in and possibly complete a real loan or reservation. Then run this cell. The suite may wait up to four hours for today's edition.

In [ ]:
RUN_LIVE_BOOK_LOAN = True

if not RUN_LIVE_BOOK_LOAN:
    print(
        "Live automation was skipped. Set RUN_LIVE_BOOK_LOAN = True in this cell "
        "only when you intend to allow a real eBiblio loan or reservation attempt."
    )
else:
    print(
        "Starting live eBiblio automation with an extended timeout. This may complete a loan or place a reservation."
    )
    live_result = subprocess.run(
        [
            sys.executable,
            "-m",
            "robot",
            "--outputdir",
            str(WORK_DIR / "results"),
            "--variable",
            "HEADLESS:True",
            "--variable",
            "PLAYWRIGHT_TIMEOUT:30s",
            str(WORK_DIR / "book-loan.robot"),
        ],
        cwd=WORK_DIR,
        check=False,
    )
    if live_result.returncode != 0:
        raise RuntimeError(
            f"Robot Framework live run failed with exit code {live_result.returncode}. "
            "Inspect the generated report and log in the private runtime."
        )
    print(f"Run artifacts were written to {WORK_DIR / 'results'}.")

## 9. Extract last result from output.xml

In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path

# Buscar el archivo output.xml en el directorio local o en Drive
output_path = Path("/content/results/output.xml")
if not output_path.exists():
    gdrive_root = globals().get('WORK_DIR', Path('/content/drive/MyDrive/ebiblio-book-loan'))
    output_path = gdrive_root / "results" / "output.xml"

print(f"Analizando archivo de resultados en: {output_path}\n")

if output_path.exists():
    try:
        tree = ET.parse(output_path)
        root = tree.getroot()

        # Buscar todos los mensajes con estado FAIL en el XML
        failures = []
        for msg in root.iter('msg'):
            if msg.attrib.get('level') == 'FAIL':
                failures.append(msg.text)

        # Alternativa: Buscar mensajes de error generales de estado
        for status in root.iter('status'):
            if status.attrib.get('status') == 'FAIL' and status.text:
                failures.append(status.text)

        if failures:
            print("❌ ERRORES DE EJECUCIÓN DETECTADOS:\n")
            for i, error_msg in enumerate(set(failures), 1):
                print(f"  {i}. {error_msg.strip()}")
        else:
            print("No se encontraron mensajes 'FAIL' explícitos en el XML. Comprueba si el archivo está vacío o incompleto.")

    except Exception as e:
        print(f"⚠ Error al procesar el archivo XML: {e}")
else:
    print("❌ No se encontró el archivo output.xml. Asegúrate de que la prueba se haya ejecutado previamente.")

## 10. Optional: Network diagnosis from Colab

In [ ]:
import subprocess
import requests

print("=== DIAGNÓSTICO DE RED DESDE COLAB ===\n")

target_url = "https://madrid.ebiblio.es/"

# Prueba 1: DNS Lookup
print("1. Intentando resolver DNS de madrid.ebiblio.es...")
try:
    import socket
    ip = socket.gethostbyname("madrid.ebiblio.es")
    print(f"   ✓ Resuelto con éxito. IP: {ip}\n")
except Exception as e:
    print(f"   ❌ Error al resolver DNS: {e}\n")

# Prueba 2: Ping / Conectividad con curl básico
print("2. Intentando descargar cabeceras con curl (límite de 10 segundos)...")
try:
    res = subprocess.run(
        ["curl", "-I", "-s", "--max-time", "10", target_url],
        capture_output=True, text=True
    )
    if res.returncode == 0:
        print("   ✓ Respuesta de curl exitosa:")
        print(res.stdout)
    else:
        print(f"   ❌ Curl falló con código {res.returncode}.")
        print(f"   Error: {res.stderr}")
except Exception as e:
    print(f"   ❌ Error al ejecutar curl: {e}\n")

# Prueba 3: Petición HTTP con User-Agent de navegador real
print("\n3. Intentando petición HTTP simulada con requests (User-Agent real)...")
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}
try:
    response = requests.get(target_url, headers=headers, timeout=15)
    print(f"   ✓ Respuesta HTTP {response.status_code} recibida correctamente.")
    print(f"   Longitud del contenido: {len(response.text)} bytes")
except requests.exceptions.Timeout:
    print("   ❌ Error: La petición excedió el límite de tiempo (Timeout de 15s).")
    print("      Esto confirma que el firewall de eBiblio está bloqueando o ignorando el tráfico de Google Colab.")
except Exception as e:
    print(f"   ❌ Error de conexión: {e}")

## 11. View private run reports

After a live run, these links open the Robot Framework report and log in the Colab runtime. They may contain account or session details; do not share them or save them in a shared notebook.

In [ ]:
import shutil
from pathlib import Path
from IPython.display import FileLink, display, HTML

# Obtener la ruta de trabajo actual preferiblemente desde el entorno de Google Drive
gdrive_root = globals().get('WORK_DIR', Path('/content/drive/MyDrive/ebiblio-book-loan'))
gdrive_results = gdrive_root / "results"

# Destino temporal local en Colab para poder generar enlaces descargables y visualizables
local_results = Path("/content/results")
local_results.mkdir(parents=True, exist_ok=True)

report_names = ("report.html", "log.html")
reports_copied = False

print("=== REPORTES DE LA AUTOMATIZACIÓN ===\n")
for name in report_names:
    gdrive_file = gdrive_results / name
    local_file = local_results / name
    if gdrive_file.is_file():
        shutil.copy2(gdrive_file, local_file)
        print(f"Archivo: {name}")
        print(f"  - Guardado permanente: {gdrive_file}")
        print("  - Enlace de descarga local en Colab:")
        display(FileLink(str(local_file)))
        print("-" * 50)
        reports_copied = True

# Enlace directo para abrir la carpeta de Google Drive en la web buscando por el nombre de la carpeta
if gdrive_results.exists():
    # El enlace con búsqueda automatiza ir directamente a la carpeta sin tener que navegar a mano
    drive_web_url = "https://drive.google.com/drive/u/0/search?q=parent:root%20name:%27ebiblio-book-loan%27"
    # Si está dentro de MyDrive generamos el enlace de búsqueda por la carpeta de resultados directamente
    results_search_url = "https://drive.google.com/drive/u/0/search?q=name:%27results%27"

    display(HTML(f'''
    <h3>📂 Enlaces directos a tu Google Drive:</h3>
    <ul>
        <li><a href="{drive_web_url}" target="_blank" style="font-size:16px; font-weight:bold; color:#1a73e8;">👉 Abrir carpeta principal "ebiblio-book-loan" en Google Drive</a></li>
        <li><a href="{results_search_url}" target="_blank" style="font-size:16px; font-weight:bold; color:#1a73e8;">👉 Buscar carpeta de "results" directamente en tu Google Drive</a></li>
    </ul>
    '''))

if not reports_copied:
    print(f"No se encontraron reportes en la carpeta de Google Drive: {gdrive_results}. Asegúrate de haber ejecutado la automatización en vivo primero.")

## 12. Cleanup Temporary Files
Run this cell to delete the sensitive credential files and execution reports from your Google Drive after you have finished.

In [ ]:
import shutil
from pathlib import Path

# Confirmación para realizar la limpieza
CONFIRM_DELETE = True

if CONFIRM_DELETE:
    # 1. Limpieza en Google Drive
    if 'WORK_DIR' in globals() and WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
        print(f"Se ha eliminado el directorio de Google Drive: {WORK_DIR}")
    else:
        print("El directorio de Google Drive ya estaba limpio o no existe.")

    # 2. Limpieza del entorno local de Colab
    local_patterns = ["*book-loan*.robot", "*credentials*.resource"]
    local_content_dir = Path('/content')
    deleted_local_count = 0

    # Eliminar archivos .robot y .resource sueltos en /content
    for pattern in local_patterns:
        for f in local_content_dir.glob(pattern):
            if f.is_file():
                f.unlink()
                deleted_local_count += 1

    # Eliminar carpetas locales de resultados si existen
    local_results = Path('/content/results')
    if local_results.exists():
        shutil.rmtree(local_results)
        print("Se ha eliminado el directorio de resultados local: /content/results")

    local_project = Path('/content/book-loan')
    if local_project.exists():
        shutil.rmtree(local_project)
        print("Se ha eliminado el directorio de proyecto local: /content/book-loan")

    print(f"Limpieza local completada ({deleted_local_count} archivos eliminados de /content).")
else:
    print("Limpieza cancelada. Establece CONFIRM_DELETE = True para borrar los ficheros.")